# Sweep 12b — Notes Thesis: Note Configuration Study

**~27 runs ≈ 4–5 hours**

| Group | What | Runs |
|-------|------|------|
| **D** | Note projection dim: 64 / 128 / 256 | 9 |
| **E** | Embedding model: ClinicalBERT raw vs Groq-intent vs Groq-phenotype | 9 |
| **F** | CAMO on vs off | 6 |
| **G** | Best combo (optimal from D+E+F) | 3 |

**Requires:** Run 12a first — paste A0 and A1 Jaccard values into the results compiler cell.

All configs: naked + notes (no labs, no copy), hgt_ehr_feat backbone.

**Group E note:** Both Groq pkls are COMPLETE (14,699/14,699 notes encoded).  
Groq was used as a *summarizer* → ClinicalBERT then encodes the summaries.
- `groq_intent` (approach A): intent-focused prompt — what does this patient need?
- `groq_phenotype` (approach B): phenotype-focused prompt — what conditions does this patient have?

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn

In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

print("Setup done:", os.getcwd())

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

BACKBONE_MODEL = {
    "graph_encoder_type": "hgt_ehr_feat",
    "hgt_layers": 1,
    "pos_weight_cap": 15.0,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
)

# Base: naked + notes, no labs, no copy
BASE_MODEL_OV = {"copy_mechanism": False, **BACKBONE_MODEL}
BASE_PREP_OV  = {"lab_dim": 0}  # note_method stays default (clinicalbert_chunk_pool)

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_12b_notes_config")
results_log = []

print("Config helpers ready.")

In [ ]:
import subprocess
from pathlib import Path

Path("smoke_s12b").mkdir(exist_ok=True)

SMOKE = [
    # Group D — note_proj_dim
    {"name": "D_proj64",
     "model_ov": {**BASE_MODEL_OV, "note_proj_dim": 64},
     "prep_ov": BASE_PREP_OV, "flags": ""},
    # Group E — Groq embeddings
    {"name": "E_groq_intent",
     "model_ov": BASE_MODEL_OV, "prep_ov": BASE_PREP_OV,
     "flags": "--note_pkl data/processed/note_embeddings_groq_intent.pkl"},
    {"name": "E_groq_phenotype",
     "model_ov": BASE_MODEL_OV, "prep_ov": BASE_PREP_OV,
     "flags": "--note_pkl data/processed/note_embeddings_groq_phenotype.pkl"},
    # Group F — CAMO
    {"name": "F_camo",
     "model_ov": BASE_MODEL_OV, "prep_ov": BASE_PREP_OV,
     "flags": "--use_camo"},
]

smoke_results = []
for t in SMOKE:
    cfg_path = f"s12b_smoke_{t['name']}.yaml"
    make_config(cfg_path, model_overrides=t["model_ov"],
                preprocessing_overrides=t["prep_ov"], smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {t['flags']} "
           f"--seed 42 --results_dir smoke_s12b/{t['name']}")
    print(f"SMOKE {t['name']}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (proc.stdout + proc.stderr).strip().split("\n")[-5:]
    for l in tail: print(l)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {t['name']}")
    print(f">>> {status}\n")

for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")

## Group D — Note projection dimension

`note_proj_dim` controls the size of the projection from 768d ClinicalBERT → scoring head.  
Default is `null` → `hidden_dim // 2 = 128`.

- D0: 64 (compressed, fewer params)
- D1: 128 (default)
- D2: 256 (same size as hidden_dim — full expressiveness)

**Key question:** Is 128 a bottleneck, or is the projection already wide enough?

In [ ]:
import subprocess, gc, torch

GROUP_D = [
    {"name": "D0_proj64",  "proj": 64,   "note": "note_proj_dim=64  (compressed)"},
    {"name": "D1_proj128", "proj": 128,  "note": "note_proj_dim=128 (default = hidden_dim//2)"},
    {"name": "D2_proj256", "proj": 256,  "note": "note_proj_dim=256 (= hidden_dim, full expressiveness)"},
]

for cfg in GROUP_D:
    print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
    cfg_path = f"s12b_{cfg['name']}.yaml"
    make_config(cfg_path,
                model_overrides={**BASE_MODEL_OV, "note_proj_dim": cfg["proj"]},
                preprocessing_overrides=BASE_PREP_OV)
    for seed in SEEDS:
        run_name = f"{cfg['name']}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{BACKBONE_FLAGS} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n=== {run_name} ===\n>> {cmd}\n")
        try:
            with open(run_dir / "training_log.txt", "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Group D done.")

## Group E — Note embedding model

All three pkls are **fully computed** (14,699/14,699 notes, 768d ClinicalBERT encoding).

| Variant | How generated | File |
|---------|--------------|------|
| E0 ClinicalBERT | Raw chunk+pool on full discharge notes | `note_embeddings_mimic3.pkl` |
| E1 Groq-intent | LLaMA summarizes with intent prompt → ClinicalBERT | `note_embeddings_groq_intent.pkl` |
| E2 Groq-phenotype | LLaMA summarizes with phenotype prompt → ClinicalBERT | `note_embeddings_groq_phenotype.pkl` |

**Key question:** Does an LLM-compressed, structured summary extract more signal than raw chunk-pooling?  
Intent prompt ("what does this patient need?") vs phenotype prompt ("what conditions does this patient have?") also tests which summarization framing aligns better with drug recommendation.

In [ ]:
import subprocess, gc, torch

GROUP_E = [
    {
        "name": "E0_clinicalbert_raw",
        "flags": "",
        "note": "ClinicalBERT chunk+pool on full notes (default)",
    },
    {
        "name": "E1_groq_intent",
        "flags": "--note_pkl data/processed/note_embeddings_groq_intent.pkl",
        "note": "Groq LLaMA intent summary → ClinicalBERT (approach A: what does this patient need?)",
    },
    {
        "name": "E2_groq_phenotype",
        "flags": "--note_pkl data/processed/note_embeddings_groq_phenotype.pkl",
        "note": "Groq LLaMA phenotype summary → ClinicalBERT (approach B: what conditions does this patient have?)",
    },
]

cfg_path = "s12b_group_e.yaml"
make_config(cfg_path, model_overrides=BASE_MODEL_OV, preprocessing_overrides=BASE_PREP_OV)

for cfg in GROUP_E:
    print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
    for seed in SEEDS:
        run_name = f"{cfg['name']}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{BACKBONE_FLAGS} {cfg['flags']} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n=== {run_name} ===\n>> {cmd}\n")
        try:
            with open(run_dir / "training_log.txt", "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Group E done.")

## Group F — CAMO

CAMO is a contrastive/augmentation mechanism for the notes channel.  
Tests whether it improves note signal on the naked+notes backbone.

In [ ]:
import subprocess, gc, torch

GROUP_F = [
    {"name": "F0_no_camo", "flags": "",          "note": "No CAMO (default)"},
    {"name": "F1_camo",    "flags": "--use_camo", "note": "CAMO enabled"},
]

cfg_path = "s12b_group_f.yaml"
make_config(cfg_path, model_overrides=BASE_MODEL_OV, preprocessing_overrides=BASE_PREP_OV)

for cfg in GROUP_F:
    print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
    for seed in SEEDS:
        run_name = f"{cfg['name']}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{BACKBONE_FLAGS} {cfg['flags']} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n=== {run_name} ===\n>> {cmd}\n")
        try:
            with open(run_dir / "training_log.txt", "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Group F done.")

## Group G — Best combo

After reviewing D/E/F results above, set the best options below and run 3 seeds.

In [ ]:
# ── EDIT THESE based on D/E/F results ─────────────────────────────────────────
# Best note projection dim (64, 128, or 256)
BEST_PROJ_DIM = 128
# Best note embedding: "", "--note_pkl data/processed/note_embeddings_groq_intent.pkl",
#                      or "--note_pkl data/processed/note_embeddings_groq_phenotype.pkl"
BEST_NOTE_PKL_FLAG = ""
# Use CAMO if F1 > F0 + 0.002
USE_CAMO = False
# Use hist notes if C1 > C0 + 0.001 (from 12a Group C)
USE_HIST_NOTES = False
# ─────────────────────────────────────────────────────────────────────────────

best_flags = BEST_NOTE_PKL_FLAG
if USE_CAMO:       best_flags += " --use_camo"
if USE_HIST_NOTES: best_flags += " --use_hist_notes"
best_flags = best_flags.strip()

print(f"Best note config:")
print(f"  note_proj_dim = {BEST_PROJ_DIM}")
print(f"  flags = '{best_flags}'")

import subprocess, gc, torch

cfg_path = "s12b_G_best_combo.yaml"
make_config(cfg_path,
            model_overrides={**BASE_MODEL_OV, "note_proj_dim": BEST_PROJ_DIM},
            preprocessing_overrides=BASE_PREP_OV)

for seed in SEEDS:
    run_name = f"G_best_combo_seed{seed}"
    run_dir  = reports_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {best_flags} "
           f"--seed {seed} --results_dir {run_dir}")
    print(f"\n=== {run_name} ===\n>> {cmd}\n")
    try:
        with open(run_dir / "training_log.txt", "w") as lf:
            proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end="")
                lf.write(line)
            proc.wait()
        status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
        results_log.append(f"{status}: {run_name}")
    except Exception as e:
        results_log.append(f"CRASH: {run_name}: {e}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nAll groups done — {len(results_log)} runs | {sum(1 for r in results_log if 'SUCCESS' in r)} succeeded")

In [ ]:
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_12b_notes_config")

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    run_name = jp.parent.name
    idx = run_name.rfind("_seed")
    if idx == -1: continue
    results.setdefault(run_name[:idx], []).append(d)

# Fill these in from 12a results
JAC_A0_NAKED    = 0.0  # <-- from 12a A0
JAC_A1_NOTES    = 0.0  # <-- from 12a A1

def summarize(name):
    runs = results.get(name, [])
    if not runs: return None
    jac = [get_metric(d, "jaccard") for d in runs]
    return {"jac_mean": np.mean(jac), "jac_std": np.std(jac, ddof=1) if len(jac)>1 else 0.0, "n": len(jac)}

def row(name, label, ref_jac):
    s = summarize(name)
    if not s:
        print(f"  {label:<36} (missing)")
        return 0.0
    print(f"  {label:<36} {s['jac_mean']:.4f}±{s['jac_std']:.4f}  Δ={s['jac_mean']-ref_jac:+.4f}  n={s['n']}")
    return s["jac_mean"]

REF = JAC_A1_NOTES  # reference: naked+notes (no labs, no copy)
print(f"Reference — A1 notes only: {REF:.4f}  (from 12a)")

print("\n" + "="*75)
print("GROUP D — Note projection dimension")
print("="*75)
jac_d0 = row("D0_proj64",  "proj_dim=64",  REF)
jac_d1 = row("D1_proj128", "proj_dim=128 (default)", REF)
jac_d2 = row("D2_proj256", "proj_dim=256", REF)
best_proj = max([(jac_d0,64),(jac_d1,128),(jac_d2,256)], key=lambda x: x[0])
print(f"  → Best projection dim: {best_proj[1]}  (Jac={best_proj[0]:.4f})")

print("\n" + "="*75)
print("GROUP E — Note embedding model")
print("="*75)
jac_e0 = row("E0_clinicalbert_raw",  "ClinicalBERT raw",        REF)
jac_e1 = row("E1_groq_intent",       "Groq intent summary",     REF)
jac_e2 = row("E2_groq_phenotype",    "Groq phenotype summary",  REF)
best_emb_jac = max(jac_e0, jac_e1, jac_e2)
best_emb_name = ["ClinicalBERT raw","Groq intent","Groq phenotype"][[jac_e0,jac_e1,jac_e2].index(best_emb_jac)]
print(f"  → Best embedding: {best_emb_name}  (Jac={best_emb_jac:.4f})")
if jac_e1 > jac_e0 + 0.002 or jac_e2 > jac_e0 + 0.002:
    print(f"  LLM summarization HELPS: structured summary > raw chunk-pool")
else:
    print(f"  Raw ClinicalBERT is competitive: LLM summarization adds minimal benefit")

print("\n" + "="*75)
print("GROUP F — CAMO")
print("="*75)
jac_f0 = row("F0_no_camo", "No CAMO",       REF)
jac_f1 = row("F1_camo",    "CAMO enabled",  REF)
print(f"  Δ_camo: {jac_f1-jac_f0:+.4f}")

print("\n" + "="*75)
print("GROUP G — Best combo")
print("="*75)
jac_g = row("G_best_combo", "Best combo", JAC_A0_NAKED)
print(f"  Δ vs naked: {jac_g - JAC_A0_NAKED:+.4f}")
print(f"  Δ vs baseline notes: {jac_g - JAC_A1_NOTES:+.4f}")

print("\n" + "="*75)
print("OPTIMAL NOTE CONFIG RECOMMENDATION")
print("="*75)
print(f"  note_proj_dim: {best_proj[1]}")
print(f"  note embedding: {best_emb_name}")
print(f"  CAMO: {'yes' if jac_f1 > jac_f0 + 0.002 else 'no benefit'}")

In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_12b_notes_config.zip"
rd = Path("experiment_reports/sweep_12b_notes_config")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")): zf.write(p, p.relative_to(rd))
    for p in sorted(rd.rglob("training_log.txt")): zf.write(p, p.relative_to(rd))
print(f"Exported → {zip_name}")